# Telco credit line - PD model

**What this does.** Takes the two tables built by `sql/42_model_datasets.sql`,
splits the labelled one 70/20/10, fits a probability-of-default model, tests it
on rows nothing was tuned on, measures how far the scoring population has
drifted, then scores the live set and produces the two tables the limit engine
needs.

### The target

| | |
|---|---|
| **Screen** | revenue at or above the window's bar in **2 or more** of 6 months, never one-way barred, never two-way barred |
| **Label** | `y = 1` if **two-way barred** (`sbrp_stat_id = 4`) within the **4 months immediately after** |

### The two tables

Data runs `140301..140506`. **1403 is deliberately unused** - revenue per
subscriber-month grew 1.86x from 1403 to 1405, so those months describe a
different population.

| table | features | label |
|---|---|---|
| `dcb_model` | 140401..140406 | 140407..140410 |
| `dcb_score` | 140501..140506 | none, the outcome is the future |

**Measured:** this window returned **5,100,390** subscribers at **0.5541%** in
`43_cohort_funnel.sql` D5, at the production bar. `dcb_model` uses a *lower*
bar, so expect more rows and a somewhat higher rate.

### Why these windows

Both feature windows are months 1-6 of their year, and the label window is
months 7-10. Two things follow:

- the feature windows sit at the **same seasonal position**, so their
  distributions are comparable;
- `dcb_score`'s features end at 140506, so a 4-month line is really exposed over
  `140507..140510` — **also months 7-10**. The model is trained on the season it
  will actually face.

That second point is not cosmetic. `43_cohort_funnel.sql` measured the two-way
bar rate at **0.5730%** and **0.5541%** for labels on months 7-10, against
**0.9492%** for a label on months 1-4 — a factor of **1.68** — while screen
breadth moved the rate not at all over the same comparison. Jalali month 1 is
Farvardin and carries Nowruz.

### The 70/20/10 split

`dcb_model` is one table, split here on a hash of `sbrp_id`:

| split | share | used for |
|---|---|---|
| `TRAIN` | 70% | fit both models |
| `VALID` | 20% | choose between them, and fit the calibrator |
| `TEST` | 10% | **read once, at the end.** Nothing is tuned on it. |

### What a random split cannot tell you

A split by subscriber measures generalisation to **other subscribers in the same
period**. It cannot measure generalisation to a **later period** — and a later
period is exactly where this model gets applied.

That matters more than usual here, because the drift is **measured, not
guessed**: revenue per subscriber-month is up **1.64x at the p90** between the
two windows, and the fixed 170,000 Toman screen admits 14.6% of the base in one
and 24.4% in the other.

So **TEST AUC is an upper bound on live performance, not an estimate of it.**
Two things are done about that rather than noted and ignored:

1. **Section 5** measures feature drift between the two tables directly — the
   part a random split hides.
2. Each level feature has a **relative twin** from the SQL, divided by its own
   window's median. Raw Rial levels drift; a ratio to the window median does
   not.

### How to run

1. Run `sql/42_model_datasets.sql`, check `T1` and `T2`.
2. Export `dcb_model` and `dcb_score` into `notebook/data/` (parquet or CSV,
   one file or `_part*` pieces — the loader handles all of those).
3. Run this notebook top to bottom, **in order**. Each cell depends on names
   defined above it.

---
## 0. Imports

Nothing here touches data. The versions are printed because one step in this
pipeline is version-sensitive and it is worth seeing the numbers before a
long fit fails on an API change.

In [ ]:
import gc
import os
import glob
import time
import hashlib        # the split and the down-sampling draw; section 3 says why

import numpy as np
import pandas as pd
import sklearn

print("pandas ", pd.__version__)
print("numpy  ", np.__version__)
print("sklearn", sklearn.__version__)

# These are Rial and Toman figures in the billions; 6.9e+06 is not a readable
# subscriber count.
pd.set_option("display.float_format", lambda v: f"{v:,.6f}")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

---
## 1. Configuration

Every knob in one place; the only cell to edit when re-running.

`TICKET_TOMAN` affects **no fit and no metric** — it only converts the PD
ranking into money columns in section 11, and it is the business's number.

`USE_RELATIVE_ONLY` is the drift switch. Section 5 tells you whether to set it.

In [ ]:
DATA       = "data"
MODEL_FILE = "dcb_model"      # features 140401..140406, label 140407..140410
SCORE_FILE = "dcb_score"      # features 140501..140506, no label
OUTDIR     = "outputs"

GOOD_KEEP  = 10               # keep this many goods out of 100, in TRAIN only
SEED       = 42

SPLIT_TRAIN = 70              # hash mod 100 under this -> train
SPLIT_VALID = 90              # under this -> valid; at or over -> test

# EXPECTED SHAPE, from 42's T1. Ported from the earlier notebook's CFG, which
# had this right: "a missing part is otherwise undetectable - from the outside
# it looks exactly like a smaller export". The duplicate-sbrp_id check catches
# a part counted TWICE; nothing catches a part that never arrived, and a
# half-loaded training set produces a perfectly plausible model on half the
# data. The row count is the stronger form - it also catches a TRUNCATED
# export, which a part count does not.
#
# Set a value to None to skip that check, and update these numbers whenever
# 42 is re-run with a different bar.
# rows is None because the DCB label changed the row count and the new one is
# not known until 42 runs. The INNER JOIN to lab now requires presence in
# 140408 alone rather than in any of 140407-140410, so dcb_model loses the
# subscribers who appear in the window but not in that month. The parts check
# is the one that catches a half-finished export; compare the printed row
# count against T1 in 42 by eye.
# parts is per EXTENSION, because the two routes give different file counts.
# The export writes TWO parquet parts; the usual next step is to concatenate
# them into ONE csv and train from that. Both are correct, and a single flat
# number would reject whichever one was not used.
EXPECT = {
    "dcb_model": dict(rows=None, parquet_parts=2, csv_parts=1),
    "dcb_score": dict(rows=None, parquet_parts=2, csv_parts=1),
}

# The DCB ticket. The product is one-month credit, 100,000 minimum, 300,000
# average. This drives the exposure and expected-loss columns only - it does
# not touch the model.
TICKET_TOMAN = 300_000

# The ONE-MONTH rate, measured by 52_one_month_label_choice.sql D4: 19,215
# two-way bars in 8,701,085 screened = 0.2208 pct. The old value here was
# 0.5541 pct, which belongs to a FOUR-month label - leaving it would make a
# correct run look 60 pct off.
#
# A rate near 0.55 pct would mean the label window has quietly widened back to
# four months; near 0.95 pct would mean it is also sitting in months 1-4.
REFERENCE_RATE = 0.002208

# Grade boundaries, as probabilities over the LABEL HORIZON. These were
# 0.002 / 0.005 / 0.010 / 0.020 for a four-month label. A one-month label runs
# at 0.42x of that (0.2208 against 0.5213), so the same boundaries would put
# almost everyone in grade A and leave B to E nearly empty. Rescaled.
GRADE_CUTS = [-1, 0.001, 0.002, 0.004, 0.008, 1.0]

# Raw Rial level features that have a _rel twin from the SQL. Setting the flag
# drops the raw ones and keeps the ratios - the drift-robust choice when PSI
# says the levels have moved. Section 5 will tell you.
# outst_max is deliberately absent: its raw PSI is 0.0551 (stable) while
# outst_max_rel measured 1.1404, because dividing a stable column by its
# window's near-zero median manufactures drift. The twin is no longer built.
RAW_WITH_REL = ["rev_6m", "rev_max", "paid_6m", "avail_max"]
USE_RELATIVE_ONLY = False

# WHICH FEATURES THE MODEL SEES.
#
#   "full"        every column 42 produces, Rial levels included.
#   "scale_free"  no Rial amount anywhere. Each month becomes a share of the
#                 subscriber's own 6-month total, plus payment coverage, the
#                 counts and the flags.
#
# MEASURED by validate_model.ipynb on this data: full 0.8278, scale_free
# 0.8230. The 0.0048 is worth paying. Twelve features in the full model drift
# significantly (rev_max at PSI 3.0223), so its 0.8278 is an UPPER BOUND on
# live performance; a scale-free feature cannot drift with inflation, so
# 0.8230 is an ESTIMATE of it. A bounded number traded for an unbounded one is
# the wrong way round, and the nominal design also needs re-tuning every year
# as the Rial moves.
#
# Left at "full" so a re-run reproduces what was run before. Set it to
# "scale_free" to switch.
FEATURE_SET = "full"

# DROP EVERYTHING DERIVED FROM available_credit.
#
# DCB credit is a separate pool, but the moment it is granted the subscriber's
# available_credit RISES BY THE GRANT AMOUNT. So from the second cycle onward
# avail_max, avail_avg and avail_max_rel stop describing the subscriber and
# start describing OUR OWN LENDING DECISION. The model learned their meaning on
# pre-DCB data where no such grant existed.
#
# That is a feedback loop, and the direction does not matter: whatever the
# model learned about available_credit, granting credit now moves the feature
# for exactly the subscribers we granted it to, and the next cycle re-scores
# them on a number we created. Scoring the FIRST cycle is safe because no grant
# has happened yet; every cycle after it is not.
#
# Left False so a re-run reproduces what was run before. Set it True before any
# second-cycle scoring, and compare the TEST AUC - if the cost is small, drop
# them permanently rather than carrying a feature that measures us.
DROP_AVAIL_FEATURES = False

# Not features:
#   sbrp_id         an identity; a tree would memorise individuals
#   y, cohort       the label and a constant tag
#   split           created below
#   sample_weight   created by down-sampling, not observed
#   n_label_months  built from the OUTCOME and NULL in dcb_score -> leakage
#   in_band         marks the production-equivalent rows. It is 1 for every
#                   row of dcb_score by construction, so as a feature it is
#                   the oneway_months trap again - weight in TRAIN that cannot
#                   fire at scoring time.
NOT_FEATURES = {"sbrp_id", "y", "cohort", "sample_weight", "n_label_months",
                "split", "in_band"}

os.makedirs(OUTDIR, exist_ok=True)
rng = np.random.default_rng(SEED)
print(f"outputs -> {OUTDIR}/   seed {SEED}   ticket {TICKET_TOMAN:,} Toman")
print(f"relative-only features: {USE_RELATIVE_ONLY}   feature set: {FEATURE_SET}")

---
## 2. Loading

The export may arrive as parquet or CSV, as one file or `_part1`/`_part2`
pieces, because 9M rows does not always come back from the warehouse in one
file. All of those shapes are accepted.

**The duplicate check is not decoration.** When a table is exported in parts by
a range condition, a wrong boundary gives overlapping parts; every row in the
overlap is counted twice and every rate below shifts silently. This raises
instead of continuing.

In [ ]:
def load(stem):
    """Parquet if present, else CSV, else the _part* files of either."""
    base = os.path.join(DATA, stem)
    # GLOB ONLY, no exact-name pattern. An earlier version tried
    # base + ".parquet" first and base + "*.parquet" second, but glob's "*"
    # matches the empty string, so the first was a strict subset of the second:
    # if a whole-table file AND part files both existed, the whole-table file
    # won and the parts were silently ignored - which loads stale data without
    # a word. Globbing only means that case concatenates everything and the
    # duplicate-sbrp_id check below stops the run.
    #
    # File ORDER does not matter: the parts are concatenated and
    # train/valid/test is assigned afterwards on a hash of sbrp_id.
    for ext in (".parquet", ".csv"):
        files = sorted(set(glob.glob(base + "*" + ext)))
        if not files:
            continue
        rd = pd.read_parquet if ext == ".parquet" else pd.read_csv
        parts = [rd(f) for f in files]
        df = pd.concat(parts, ignore_index=True) if len(parts) > 1 else parts[0]
        print(f"  {stem}: {len(files)} file(s) -> {len(df):,} rows "
              f"x {df.shape[1]} cols")
        if len(files) > 1:
            print(f"    {', '.join(os.path.basename(f) for f in files)}")

        exp = EXPECT.get(stem, {})
        key = "parquet_parts" if ext == ".parquet" else "csv_parts"
        want = exp.get(key)
        if want is not None and len(files) != want:
            raise ValueError(
                f"{stem}: found {len(files)} {ext} file(s), expected {want}. "
                f"A missing part looks exactly like a smaller export from the "
                f"outside, so this is the only thing that catches it. Check "
                f"the export, or set '{key}' for {stem} in EXPECT to None if "
                f"it really is in {len(files)} piece(s).")
        if exp.get("rows") is not None and len(df) != exp["rows"]:
            raise ValueError(
                f"{stem}: loaded {len(df):,} rows, expected {exp['rows']:,} "
                f"(short by {exp['rows']-len(df):,}). Either a part is missing "
                f"or truncated, or 42 was re-run with a different bar - in "
                f"which case update EXPECT from its T1 output. A short load "
                f"trains a plausible-looking model on less data than intended.")
        # THE SPLIT IS md5(str(sbrp_id)), SO THE DTYPE IS PART OF THE ANSWER.
        # str(9891234567) is '9891234567' but str(9891234567.0) is
        # '9891234567.0' - different strings, different md5, different bucket.
        # Parquet carries the integer type; a csv round trip does not, and on
        # some pandas versions a single blank cell turns the whole column to
        # float. Nothing would error: the model would simply be fitted and
        # tested on different subscribers than the parquet route gives.
        if "sbrp_id" in df.columns and not pd.api.types.is_integer_dtype(df.sbrp_id):
            before = str(df.sbrp_id.dtype)
            if df.sbrp_id.isna().any():
                raise ValueError(
                    f"{stem}: sbrp_id is {before} and contains nulls. The "
                    f"split hashes str(sbrp_id), so this cannot be made safe "
                    f"by casting - find out why an id is missing first.")
            df["sbrp_id"] = df.sbrp_id.astype("int64")
            print(f"    sbrp_id was {before}, cast to int64 - the split hashes "
                  f"its STRING form, so the type changes which subscribers "
                  f"land in train, valid and test")

        if "sbrp_id" in df.columns:
            dup = len(df) - df.sbrp_id.nunique()
            if dup:
                raise ValueError(
                    f"{dup:,} duplicate sbrp_id across "
                    f"{', '.join(os.path.basename(f) for f in files)}. "
                    f"Those subscribers are counted twice and every metric "
                    f"below would be wrong. Two causes: a whole-table export "
                    f"left in the directory beside its own parts, or parts cut "
                    f"on a boundary that overlaps. Delete the extra file, or "
                    f"re-split with sql/47_export_parts.sql and check its V1.")
        return df
    # Nothing matched. A bare "not found" sends you hunting, so say what the
    # glob looked for and what is actually in the directory - the usual cause
    # is a file carrying an older stem, such as dcbtrain or dcbs_scoreset from
    # the previous pipeline, which the glob cannot match.
    tried = [os.path.join(DATA, stem) + "*" + e for e in (".parquet", ".csv")]
    here = []
    if os.path.isdir(DATA):
        here = sorted(f for f in os.listdir(DATA)
                      if f.endswith((".parquet", ".csv")))
    lines = [f"nothing found for {stem}.",
             f"  looked for : {' , '.join(tried)}",
             f"  DATA={DATA!r} resolves to {os.path.abspath(DATA)}",
             f"  cwd        : {os.getcwd()}"]
    if here:
        lines.append(f"  data files present but NOT matching: {here}")
        lines.append(f"  -> only the STEM matters. Rename so the file begins "
                     f"'{stem}'; any suffix works, e.g. {stem}_even, "
                     f"{stem}_odd, {stem}_p1.")
    elif os.path.isdir(DATA):
        lines.append(f"  {DATA}/ exists but holds no .parquet or .csv at all.")
    else:
        lines.append(f"  {DATA}/ does not exist from this cwd - the notebook "
                     f"may be running from a different directory.")
    lines.append("  Build the tables with sql/42_model_datasets.sql; "
                 "sql/47_export_parts.sql splits them if needed.")
    raise FileNotFoundError("\n".join(lines))


print("LOADING")
md_df = load(MODEL_FILE)
sc    = load(SCORE_FILE)

shared  = set(md_df.columns) & set(sc.columns)
only_md = sorted(set(md_df.columns) - shared)
only_sc = sorted(set(sc.columns) - shared)
print(f"\n  columns only in dcb_model: {only_md}")
print(f"  columns only in dcb_score: {only_sc}")
print("  (nothing, or only y/n_label_months, is what you want here - anything")
print("   else means one table came from an older run of the SQL)")

---
## 3. The 70/20/10 split

The draw is on **`md5(sbrp_id)`**, never on `sbrp_id` itself. Every observed id
in this base is **odd**, and they share a five-digit block, so `MOD` on the raw
id selects a structured slice of the network rather than a sample. A parity
split on `sbrp_id` once put all 11.5M rows in one half of a two-way split.

Hashing also makes the split **reproducible without storing it** — the same
subscriber lands in the same split on every re-run, and in any other script
using the same rule. `T3` in the SQL previews exactly this draw.

Two checks follow the split:

- the whole table's rate against the **0.5541%** on record for this window;
- the three splits' rates against each other. They are random draws from one
  table, so a real gap would mean the hash is misbehaving.

In [ ]:
def hash_bucket(ids):
    """md5(sbrp_id) mod 100 - see the markdown above for why not MOD on the id."""
    return ids.astype(str).map(
        lambda s: int(hashlib.md5(s.encode()).hexdigest()[:8], 16) % 100)


h = hash_bucket(md_df.sbrp_id).to_numpy()
md_df["split"] = np.where(h < SPLIT_TRAIN, "train",
                 np.where(h < SPLIT_VALID, "valid", "test"))

print("SPLITTING dcb_model 70/20/10 on a hash of sbrp_id")
for nm in ("train", "valid", "test"):
    d = md_df[md_df.split == nm]
    print(f"  {nm:<6} {len(d):>10,} rows  {int(d.y.sum()):>7,} bad  "
          f"{d.y.mean():.4%}")

rate = md_df.y.mean()
print(f"\n  whole table {len(md_df):,} rows at {rate:.4%}")
print(f"  reference for this window at the production bar "
      f"{REFERENCE_RATE:.4%} -> {abs(rate-REFERENCE_RATE)/REFERENCE_RATE:.1%} apart")
print("  (the bar here is LOWER than production, so a somewhat higher rate is")
print("   expected - the extra subscribers admitted are the poorer ones)")

# The threshold was 0.0080, set against a FOUR-month label. Tying it to
# REFERENCE_RATE means it keeps working when the horizon changes again.
if rate > 2.0 * REFERENCE_RATE:
    print(f"\n  WARNING the rate is {rate/REFERENCE_RATE:.1f}x the reference for this")
    print("  horizon. The usual cause is a label window wider than intended -")
    print("  check TERM_MONTHS and the label months in 42_model_datasets.sql")
    print("  before going further.")

rates = [md_df[md_df.split == nm].y.mean() for nm in ("train", "valid", "test")]
if max(rates) / max(min(rates), 1e-12) > 1.5:
    print(f"\n  WARNING split event rates differ by more than 1.5x: "
          f"{[f'{r:.4%}' for r in rates]} - check the hash")
else:
    print("\n  the three splits carry consistent event rates")

---
## 3b. The band — which rows the metrics are read on

`MODEL_POP` in `sql/42_model_datasets.sql` decides what `dcb_model` contains:

| `MODEL_POP` | `dcb_model` holds |
|---|---|
| `screened` | the production-equivalent population (default) |
| `superset` | a lower revenue bar, same bar-history filters |
| `all` | every subscriber in the window |

**Whatever it contains, the metrics are read on `in_band = 1` rows only**,
because that is the only population `dcb_score` holds and therefore the only
one that ever gets scored.

This is not a technicality. Measured on synthetic data, a model trained on the
whole base reported an AUC of **0.78–0.82 on the whole base** while performing
at **0.63–0.69 in the band**. Comparing a whole-base AUC against a band AUC
compares two different test sets and tells you nothing — except that the
flattering number is the one you would be tempted to quote.

The same experiment, AUC always measured in the band:

| | train on band | train on superset | train on whole base |
|---|---|---|---|
| risk relationship same across the range | 0.6835 | 0.6839 | 0.6838 |
| relationship differs inside the band | 0.7235 | 0.7127 | **0.6342** |

So `all` is break-even at best and costs 0.09 at worst; `superset` stays within
0.011 of band-only in both worlds and buys robustness to the screen drifting
(it sits at 24.4% of the base now and was 14.6% a year ago).

**W4 in `44_where_are_the_bads.sql` is the free leading indicator** for which
of those two worlds the real data is in: if the bad rate still falls across
`rev_months` inside the cohort, the relationship survives the screen.

In [ ]:
if "in_band" not in md_df.columns:
    md_df["in_band"] = 1
    print("  no in_band column - rebuild with the current SQL. Assuming every")
    print("  row is in band, which is only true for MODEL_POP=screened.")

band_share = md_df.in_band.mean()
print(f"  in_band rows: {int(md_df.in_band.sum()):,} of {len(md_df):,} "
      f"({band_share:.1%})")
if band_share < 0.999:
    print("\n  dcb_model is WIDER than the production screen, so TRAIN sees")
    print("  rows that can never be scored. Metrics below are reported both")
    print("  ways, and only the in-band figures are comparable across runs.")
else:
    print("\n  dcb_model is the production-equivalent population, so in-band")
    print("  and all-rows metrics are the same thing here.")

---
## 4. Choosing the features

Three filters, then the guards.

**Filters.** Drop the non-features; keep only columns present in *both* tables
(a column missing from `dcb_score` cannot be used at scoring time); keep only
numeric columns.

**Zero variance.** `oneway_months` is zero for every row *by construction* —
the screen requires it. A constant column is dead weight at best. At worst it is
the trap `debt_scr` and `suspend_scr` set earlier in this project: both all-zero
in 6 of 10 months, producing a PSI of 13.89 and 14.41 that read as catastrophic
drift when it was really a loading gap.

**The leakage assertion** is cheap, and it catches the one class of mistake that
produces a beautiful AUC and a worthless model.

In [ ]:
def add_scale_free(d):
    """Every Rial column re-expressed against the subscriber's own 6-month total.

    Scale-free BY CONSTRUCTION: no window median, so unlike a _rel twin it
    cannot manufacture drift when the median sits near zero - which is what
    put outst_max_rel at PSI 1.1404 against its raw column's 0.0551 and got it
    removed. A share of your own total needs nothing from the window at all.
    """
    rev = d.rev_6m.to_numpy(dtype=np.float64)
    n_zero = int((rev <= 0).sum())
    den = np.where(rev > 0, rev, np.nan)       # nan, not 0 - an undefined share
    new = {}
    for i in range(1, 7):
        new[f"r{i}_sh"] = d[f"r{i}"].to_numpy() / den
        new[f"q{i}_sh"] = d[f"q{i}"].to_numpy() / den
    for nm, col in (("rev_min_sh", "rev_min"), ("rev_max_sh", "rev_max"),
                    ("rev_trend_sh", "rev_trend"), ("paid_max_sh", "paid_max"),
                    ("outst_max_sh", "outst_max"), ("outst_avg_sh", "outst_avg"),
                    ("avail_max_sh", "avail_max"), ("avail_avg_sh", "avail_avg"),
                    ("pay_cover", "paid_6m")):
        new[nm] = d[col].to_numpy() / den
    for k, v in new.items():
        d[k] = v.astype(np.float32)
    return sorted(new), n_zero


SCALE_FREE_KEEP = ["rev_months", "rev_months_wide", "pay_months", "n_arpu_null",
                   "n_zero_rev_months",
                   "n_active1", "f_reclaim", "tenure_m", "pre_months_seen",
                   "pre_ow_any", "pre_tw_any", "pre_ow_months", "pre_tw_months",
                   "pre_absent"]

if FEATURE_SET == "scale_free":
    made, nz_m = add_scale_free(md_df)
    _,    nz_s = add_scale_free(sc)
    print(f"SCALE-FREE: built {len(made)} share columns in both tables")
    if nz_m or nz_s:
        print(f"  rows with rev_6m <= 0 (every share undefined, becomes 0.0 in X): "
              f"{nz_m:,} model / {nz_s:,} score")

FEATURES = [c for c in md_df.columns if c not in NOT_FEATURES]
FEATURES = [c for c in FEATURES if c in sc.columns]
FEATURES = [c for c in FEATURES if pd.api.types.is_numeric_dtype(md_df[c])]

dead = [(c, (md_df[c].nunique(dropna=True), sc[c].nunique(dropna=True)))
        for c in FEATURES
        if max(md_df[c].nunique(dropna=True), sc[c].nunique(dropna=True)) <= 1]
if dead:
    print("DROPPED - no variance in either table:")
    for c, nun in dead:
        print(f"  {c:16s} distinct values MODEL/SCORE {nun}")
    FEATURES = [c for c in FEATURES if c not in {d[0] for d in dead}]

if DROP_AVAIL_FEATURES:
    gone = [c for c in FEATURES if c.startswith("avail_")]
    FEATURES = [c for c in FEATURES if c not in gone]
    print(f"\nDROP_AVAIL_FEATURES: removed {len(gone)} columns derived from "
          f"available_credit, which a DCB grant inflates: {gone}")

if FEATURE_SET == "scale_free":
    keep = set(SCALE_FREE_KEEP) | {c for c in FEATURES if c.endswith("_sh")} | {"pay_cover"}
    dropped_rial = [c for c in FEATURES if c not in keep]
    FEATURES = [c for c in FEATURES if c in keep]
    print(f"\nSCALE-FREE: {len(FEATURES)} features, dropped {len(dropped_rial)} "
          f"carrying a Rial level")
    print(f"  dropped: {dropped_rial}")
    leftover = [c for c in FEATURES if c.endswith("_rel")]
    assert not leftover, f"a _rel twin survived the scale-free filter: {leftover}"

if USE_RELATIVE_ONLY:
    drop = [c for c in RAW_WITH_REL if c + "_rel" in FEATURES]
    FEATURES = [c for c in FEATURES if c not in drop]
    print(f"\nUSE_RELATIVE_ONLY: dropped the raw twins {drop}")

leak = [c for c in FEATURES
        if c in ("y", "cohort", "n_label_months", "sample_weight", "split")]
assert not leak, f"label or label-derived column in the features: {leak}"
assert FEATURES, "no features survived selection"

print(f"\nFEATURES: {len(FEATURES)}")
for c in FEATURES:
    print(f"  {c}")

---
## 5. Feature drift — the part the random split hides

This is the section that matters most in this notebook, and it is the one a
70/20/10 split cannot give you.

**PSI (Population Stability Index)** compares each feature's distribution in
`dcb_model` against `dcb_score`:

| PSI | reading |
|---|---|
| under 0.10 | stable |
| 0.10 – 0.25 | moderate shift |
| over 0.25 | **significant shift** |

Expect the raw Rial levels to come back high. That is not a bug and not noise:
`43_cohort_funnel.sql` measured revenue per subscriber-month up **1.64x at the
p90** between these two windows. The `_rel` twins — each divided by its own
window's median — should come back near zero, which is the whole reason they
exist.

**If the raw levels have shifted and the relative twins have not**, set
`USE_RELATIVE_ONLY = True` in section 1 and re-run. Then compare TEST AUC: if it
holds up, prefer that model, because it will age better. A coefficient learned
on 1404 Rial means something different applied to 1405 Rial — a subscriber on
2,000,000 was upper-tail in 1404 and mid-pack in 1405, and the model would read
them as safer than their peers warrant.

Low-cardinality features (flags, small counts) are compared on their **values**
rather than on quantiles. Ten quantile bins cannot be cut from a binary column,
and an earlier version of this returned `NaN` for `f_reclaim` — a flag can drift
as easily as anything else.

In [ ]:
VALUE_PSI_MAX_CATS = 40   # at or under this many distinct values, compare on VALUES

def psi(expected, actual, bins=10):
    """Population Stability Index between two samples of one feature."""
    e = np.asarray(expected, dtype=float)
    a = np.asarray(actual,   dtype=float)
    e = e[np.isfinite(e)]
    a = a[np.isfinite(a)]
    if len(e) == 0 or len(a) == 0:
        return np.nan

    ve, ce = np.unique(e, return_counts=True)
    va, ca = np.unique(a, return_counts=True)

    # WHICH COMPARISON. Quantiles only suit a genuinely continuous column.
    # A count column is compared on its VALUES - and "few distinct values" is
    # not the same test as "few quantile bins", which is what the first
    # version of this got wrong: pre_ow_months holds 13 distinct values, so it
    # failed the len(vals) <= bins test and went to quantiles, where ~99.9 pct
    # zeros collapsed every edge onto 0 and the function returned nan. Two of
    # the six new pre_ features came back unmeasured because of it.
    use_values = len(ve) <= max(bins, VALUE_PSI_MAX_CATS)
    edges = None
    if not use_values:
        edges = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
        # Concentration collapsed the edges. That is a reason to compare on
        # values, not a reason to give up.
        use_values = len(edges) < 3

    if use_values:
        de, da = dict(zip(ve, ce)), dict(zip(va, ca))
        cats = sorted(set(de) | set(da))
        if len(cats) > 4 * VALUE_PSI_MAX_CATS:
            # continuous AND degenerate - no honest binning left
            return np.nan
        pe = np.array([de.get(v, 0) for v in cats], dtype=float)
        pa = np.array([da.get(v, 0) for v in cats], dtype=float)
    else:
        edges[0], edges[-1] = -np.inf, np.inf
        pe, _ = np.histogram(e, bins=edges)
        pa, _ = np.histogram(a, bins=edges)
        pe, pa = pe.astype(float), pa.astype(float)

    if pe.sum() == 0 or pa.sum() == 0:
        return np.nan
    pe, pa = pe / pe.sum(), pa / pa.sum()
    # a floor so an empty bin does not send the log to infinity
    pe, pa = np.clip(pe, 1e-6, None), np.clip(pa, 1e-6, None)
    return float(np.sum((pa - pe) * np.log(pa / pe)))


print("FEATURE DRIFT, dcb_model against dcb_score (PSI)")
print("  under 0.10 stable, 0.10-0.25 moderate, over 0.25 significant\n")
rows = sorted(((c, psi(md_df[c], sc[c])) for c in FEATURES),
              key=lambda t: -(t[1] if np.isfinite(t[1]) else -1))
for c, v in rows:
    flag = ("" if not np.isfinite(v) else
            "   <-- significant" if v > 0.25 else
            "   <-- moderate"    if v > 0.10 else "")
    print(f"  {c:18s} {v:8.4f}{flag}")

big = [c for c, v in rows if np.isfinite(v) and v > 0.25]
if big:
    print(f"\n  {len(big)} feature(s) shifted significantly. TEST below is an")
    print("  UPPER BOUND on live performance, not an estimate of it.")
    raw_big = [c for c in big if c in RAW_WITH_REL]
    if raw_big and not USE_RELATIVE_ONLY:
        print(f"\n  {raw_big} are raw Rial levels whose _rel twins are already")
        print("  in the data. Set USE_RELATIVE_ONLY = True in section 1, re-run,")
        print("  and compare TEST AUC.")
else:
    print("\n  no feature shifted significantly - better than the measured")
    print("  revenue drift would suggest; check the bars in the SQL are set")
    print("  per window as intended.")

---
## 6. Down-sampling TRAIN, and the weight that undoes it

At a rate near 0.6%, 99% of rows are goods and they are not where the
information is. Keeping every bad and a `GOOD_KEEP`% slice of goods makes the
fit tractable; `sample_weight` puts the base rate back.

**Only TRAIN is down-sampled.** VALID and TEST stay whole, so their metrics need
no weighting correction.

### Use `sample_weight` in every fit and every metric

Keeping 10% of goods inflates the **odds** by exactly `100 / GOOD_KEEP = 10x`,
and at these rates roughly **9x the probability** itself. Ranking survives that
distortion. **The level does not** — and the limit engine in section 11 spends
the level, not the ranking.

An unweighted Brier score on a down-sampled file read 0.29 on a 2% problem
earlier in this project, entirely from the weighting error.

The down-sampling draw is hashed with a `"ds"` prefix so it is **independent of
the train/valid/test assignment** — otherwise the two draws would correlate and
the goods kept would not be a fair slice of TRAIN.

In [ ]:
def downsample(df, keep_pct):
    """Every bad, keep_pct of the goods, with sample_weight restoring the rate."""
    bad, good = df[df.y == 1], df[df.y == 0]
    h = good.sbrp_id.astype(str).map(
        lambda s: int(hashlib.md5(("ds" + s).encode()).hexdigest()[:8], 16) % 100)
    good = good[h.values < keep_pct]
    out = pd.concat([bad, good], ignore_index=True)
    out["sample_weight"] = np.where(out.y == 1, 1.0, 100.0 / keep_pct)
    print(f"  kept every bad ({len(bad):,}) and {keep_pct} pct of goods "
          f"({len(good):,}) -> {len(out):,} rows")
    print(f"  raw rate in the sample {out.y.mean():.4%}, weighted back to "
          f"{np.average(out.y, weights=out.sample_weight):.4%}")
    return out


print("DOWN-SAMPLING TRAIN")
tr = downsample(md_df[md_df.split == "train"], GOOD_KEEP)
va = md_df[md_df.split == "valid"]
te = md_df[md_df.split == "test"]


def X(d):
    # nan_to_num because the source has real NULLs: arpu is 23.8 pct NULL, and
    # the SQL already resolves that to 0 revenue rather than minus-the-tax.
    return np.nan_to_num(d[FEATURES].to_numpy(float),
                         nan=0.0, posinf=0.0, neginf=0.0)


Xtr, ytr, wtr = X(tr), tr.y.to_numpy(int), tr.sample_weight.to_numpy(float)
Xva, yva = X(va), va.y.to_numpy(int)
Xte, yte = X(te), te.y.to_numpy(int)
wva, wte = np.ones(len(yva)), np.ones(len(yte))

# boolean masks for the production-equivalent rows - every metric below uses
# them, because dcb_score contains only in-band subscribers
bva = va.in_band.to_numpy().astype(bool)
bte = te.in_band.to_numpy().astype(bool)

print(f"\n  TRAIN {Xtr.shape}  weighted rate {np.average(ytr, weights=wtr):.4%}")
print(f"  VALID {Xva.shape}  rate {yva.mean():.4%}  "
      f"in band {bva.sum():,} at {yva[bva].mean():.4%}")
print(f"  TEST  {Xte.shape}  rate {yte.mean():.4%}  "
      f"in band {bte.sum():,} at {yte[bte].mean():.4%}")

---
## 7. Fitting: champion and challenger

**Champion — logistic regression** on standardised features. Linear, monotone in
each feature, and the coefficients can be read and argued with, which a credit
committee will want.

**Challenger — histogram gradient boosting.** Catches interactions the logistic
cannot. It is a challenger, not the default: it wins only if it wins on VALID.

`StandardScaler` matters here — the features span raw Rial revenue in the
millions and a month count in 0..6. Unscaled, `lbfgs` converges badly.

Both are fitted **with `sample_weight`**.

In [ ]:
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import StandardScaler
from sklearn.pipeline       import make_pipeline

print("FITTING champion")
t0 = time.time()
champ = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs"))
# The pipeline forwards the weight to the final step by its step name.
champ.fit(Xtr, ytr, logisticregression__sample_weight=wtr)
print(f"  logistic fitted in {time.time()-t0:,.0f}s")

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

print("FITTING challenger")
t0 = time.time()
chal = HistGradientBoostingClassifier(
    max_iter=400,              # a ceiling; early stopping usually ends sooner
    learning_rate=0.06,
    max_leaf_nodes=31,
    early_stopping=True,
    validation_fraction=0.15,  # carved out of TRAIN, never from VALID or TEST
    random_state=SEED)
chal.fit(Xtr, ytr, sample_weight=wtr)
print(f"  gradient boosting fitted in {time.time()-t0:,.0f}s, "
      f"{chal.n_iter_} iterations of a 400 ceiling")

---
## 8. Metrics, and the choice between the two models

Every metric is **weighted**.

| metric | what it tells you |
|---|---|
| **AUC / Gini** | ranking quality. This is what the limit engine consumes. |
| **KS** | separation at the best single cut. |
| **Brier** | calibration — whether the *level* is right, not just the order. |

**Accuracy is deliberately absent.** At a rate near 0.55% a model predicting
"nobody defaults" scores 99.45%. It cannot tell a working model from an empty
one.

In-sample metrics are printed for the **train-to-valid gap**, which is a useful
overfitting signal — not as grounds for choosing the model. Selection is on
**VALID**, because in-sample ranking rewards whichever model memorised TRAIN
hardest.

The decile table bins by **rank position**, not by value quantile. Isotonic
regression is a step function and produces heavy ties, so `pd.qcut` with
`duplicates="drop"` collapses bins — a first run of this gave 9 bins holding
8,090 and 433 rows, and the "furthest from 1.0" figure was then read off the
433-row bin. Rank bins are always equal-sized.

In [ ]:
def report(name, y, p, w):
    """Weighted AUC, Gini, KS and Brier."""
    from sklearn.metrics import roc_auc_score, brier_score_loss
    auc = roc_auc_score(y, p, sample_weight=w)
    br  = brier_score_loss(y, p, sample_weight=w)
    # Weighted KS: the largest gap between cumulative bads and cumulative
    # goods, walking up the score.
    o  = np.argsort(p)
    ys, ws = np.asarray(y)[o], np.asarray(w)[o]
    cb = np.cumsum(ws * ys);       cb = cb / cb[-1]
    cg = np.cumsum(ws * (1 - ys)); cg = cg / cg[-1]
    ks = np.max(np.abs(cb - cg))
    print(f"  {name:<30} AUC {auc:.4f}   Gini {2*auc-1:.4f}   "
          f"KS {ks:.4f}   Brier {br:.6f}")
    return dict(model=name, auc=auc, gini=2*auc-1, ks=ks, brier=br)


def decile_table(y, p, w, label=""):
    """Predicted against observed by score decile, weighted, equal-count bins.

    Weighted sums through agg rather than groupby.apply: apply needs
    include_groups on pandas 2.2+ and raises a TypeError without it on older
    versions, so the apply form would break on an older pandas.
    """
    pv = np.asarray(p, dtype=float)
    n  = len(pv)
    order = np.argsort(pv, kind="mergesort")
    dec = np.empty(n, dtype=int)
    dec[order] = np.arange(n) * 10 // n          # equal counts, ties and all
    d = pd.DataFrame({"d": dec,
                      "y": np.asarray(y, dtype=float),
                      "p": pv,
                      "w": np.asarray(w, dtype=float)})
    d["wp"], d["wy"] = d.w * d.p, d.w * d.y
    g = d.groupby("d").agg(n=("y", "size"), weighted_n=("w", "sum"),
                           wp=("wp", "sum"), wy=("wy", "sum"))
    g["predicted"] = g.wp / g.weighted_n
    g["observed"]  = g.wy / g.weighted_n
    g["ratio"]     = g.observed / g.predicted.replace(0, np.nan)
    # The EXPECTED bads in the decile. The ratio is not interpretable where
    # this is tiny: a decile of 2,239 rows at a predicted 0.0126 pct expects
    # 0.28 bads, so observing zero is ordinary and gives a ratio of 0.000 that
    # reads as catastrophic. An earlier version reported exactly that as
    # "furthest from 1.0".
    g["exp_bad"] = g.weighted_n * g.predicted
    g = g[["n", "weighted_n", "predicted", "observed", "exp_bad", "ratio"]]
    print(f"\n  calibration by decile{label} (ratio near 1.0 is the goal)")
    print(g.to_string(float_format=lambda v: f"{v:12.6f}"))

    MIN_EXP = 10.0
    judge = g[g.exp_bad >= MIN_EXP]
    if len(judge) == 0:
        print(f"  no decile expects {MIN_EXP:.0f}+ bads, so the ratios are all")
        print("  small-sample noise and calibration cannot be judged here.")
    else:
        worst = judge.ratio.iloc[(judge.ratio - 1.0).abs().to_numpy().argmax()]
        print(f"  furthest from 1.0: {worst:.3f}   (over the {len(judge)} of "
              f"{len(g)} deciles expecting {MIN_EXP:.0f}+ bads)")
        if len(judge) < len(g):
            print(f"  the other {len(g)-len(judge)} expect too few events for "
                  f"the ratio to mean anything")
    return g

In [ ]:
print("IN-SAMPLE (TRAIN, weighted) - flatters both; reported for the gap only")
RES = [report("logistic (champion)", ytr, champ.predict_proba(Xtr)[:, 1], wtr),
       report("HistGB (challenger)", ytr, chal.predict_proba(Xtr)[:, 1], wtr)]

In [ ]:
print("VALID - the selection set, IN BAND")
pva_c = champ.predict_proba(Xva)[:, 1]
pva_g = chal.predict_proba(Xva)[:, 1]
RV = [report("logistic (champion)", yva[bva], pva_c[bva], wva[bva]),
      report("HistGB (challenger)", yva[bva], pva_g[bva], wva[bva])]
if bva.mean() < 0.999:
    print("  and on ALL VALID rows, for contrast - NOT comparable:")
    report("logistic, all rows", yva, pva_c, wva)
    report("HistGB, all rows",   yva, pva_g, wva)

best  = "HistGB" if RV[1]["auc"] > RV[0]["auc"] else "logistic"
i     = 1 if best == "HistGB" else 0
pva   = pva_g if best == "HistGB" else pva_c
model = chal  if best == "HistGB" else champ

print(f"\n  selected on VALID AUC: {best}")
gap = RES[i]["auc"] - RV[i]["auc"]
print(f"  train-to-valid AUC gap {gap:+.4f}"
      + ("   <-- overfitted, reduce max_iter" if gap > 0.05 else "   (acceptable)"))

---
## 9. Calibration — fitted on VALID

The ranking is good but the level is not usable yet: the model was fitted on a
down-sampled file. Isotonic regression maps scores onto observed rates.

**The trap this avoids.** Fitting isotonic on a set and reading its calibration
off *the same rows* gives a ratio of 1.0000 in every decile **by construction** —
a perfect-looking table that proves nothing. An earlier version of this project
did exactly that, and the table looked flawless.

So the calibrator is fitted on **VALID** and judged on **TEST** in the next
section. The `anchor` is a residual level correction; at 1.0 isotonic needed no
rescaling, which is the healthy case. It is a **ratio, not a probability** — an
earlier version of this project printed a healthy 1.0 as "99.9993%".

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Fitted on the IN-BAND rows of VALID. Calibrating on a wider population
# would anchor the level to a base rate the scored population does not have.
iso = IsotonicRegression(out_of_bounds="clip", y_min=1e-7, y_max=1 - 1e-7)
iso.fit(pva[bva], yva[bva])
anchor = np.average(yva[bva]) / max(np.average(iso.predict(pva[bva])), 1e-9)

print(f"CALIBRATION fitted on VALID in band ({int(bva.sum()):,} rows)")
print(f"  anchor ratio {anchor:.4f}  (1.0 = no rescaling needed; a RATIO, not a PD)")

---
## 10. TEST — read once

Nothing above touched these rows. No model was chosen on them, no calibrator was
fitted on them, no threshold was tuned on them. **This is the honest
within-period number.**

Remember what it is and is not: it measures generalisation to other subscribers
**in the same period**. Section 5 is what speaks to the later period, and if it
flagged significant drift, treat this AUC as a **ceiling**.

In [ ]:
print(f"TEST - {int(bte.sum()):,} in-band rows, untouched until now")
pte = np.clip(iso.predict(model.predict_proba(Xte)[:, 1]) * anchor, 1e-7, 0.999)
RT = [report(f"{best} calibrated [TEST in band]", yte[bte], pte[bte], wte[bte])]
if bte.mean() < 0.999:
    print("  and on ALL TEST rows - NOT the scored population, shown only so")
    print("  the gap between the two is visible:")
    RT.append(report(f"{best} calibrated [TEST all rows]", yte, pte, wte))
    print(f"  the all-rows AUC runs {RT[-1]['auc']-RT[0]['auc']:+.4f} against "
          f"in-band. Quote the in-band figure.")
print(f"  mean PD {pte[bte].mean():.4%}  vs observed {yte[bte].mean():.4%}  "
      f"-> level error "
      f"{abs(pte[bte].mean()-yte[bte].mean())/max(yte[bte].mean(),1e-9):.2%}")
cal_tbl = decile_table(yte[bte], pte[bte], wte[bte], " on TEST in band")

---
## 11. Scoring the live set, and the limit tables

`dcb_score` carries the most recent six months of features and no label, because
the outcome has not happened yet. These are the subscribers the product would
lend to.

| grade | 4-month PD |
|---|---|
| A | below 0.2% |
| B | 0.2% – 0.5% |
| C | 0.5% – 1.0% |
| D | 1.0% – 2.0% |
| E | above 2.0% |

Then the two tables the limit engine consumes, ranked safest first, so row `n`
answers: *if I lend to the n safest, what is the average PD of that book?*

- **Safest N**, by share of the population. Hardcoded milestones of 1M–5M
  collapsed to a single useless row when tested on a smaller population.
- **Largest book under a PD ceiling** — the same question from the other side,
  and the form a credit committee states appetite in.

**What the money columns assume:** exposure assumes every approved subscriber
draws the **full** line, and expected loss assumes a two-way bar loses the
**whole balance** (LGD 100%). Both are deliberately pessimistic — the operator
keeps collecting after a bar, and most subscribers will not draw the full line.
Treat them as a **ceiling on the loss, not a forecast**. The PD column is the
model's output; the money columns are that PD times assumptions the business owns.

In [ ]:
print("SCORING THE LIVE SET")
psc = np.clip(iso.predict(model.predict_proba(X(sc))[:, 1]) * anchor, 1e-7, 0.999)
out = pd.DataFrame({"sbrp_id": sc.sbrp_id.values, "pd_4m": psc})
out["grade"] = pd.cut(out.pd_4m, GRADE_CUTS, labels=["A", "B", "C", "D", "E"])

# TIE-BREAK KEYS. Isotonic calibration emits a STEP function, so a large block
# of subscribers share one identical pd_4m. The book is the safest N, so the
# people sitting on the step AT the cut are admitted or refused by whatever
# order the sort happens to produce - by nothing. pandas mergesort falls back
# to CSV row order and Trino guarantees no order at all, so the same scores
# cut two ways give two different books.
#
# pay_cover is what the subscriber actually paid against what they were
# billed, over the feature window. Among subscribers the model cannot tell
# apart it is a real signal, and it is scale-free so it does not drift.
# tenure_m breaks what pay_cover ties, and sbrp_id makes the order total and
# identical in Python and SQL.
_rev = sc.rev_6m.to_numpy(dtype=float)
out["pay_cover"] = np.where(_rev > 0, sc.paid_6m.to_numpy(dtype=float) / _rev, 0.0)
out["tenure_m"]  = sc.tenure_m.to_numpy()

_n_tied = len(out) - out.pd_4m.nunique()
print(f"  distinct pd_4m values {out.pd_4m.nunique():,} over {len(out):,} subscribers"
      f"  ->  {_n_tied:,} sit on a shared value")
print("  tie-break written to the handover: pd_4m asc, pay_cover desc, tenure_m desc, sbrp_id asc")

print(f"  scored {len(out):,} subscribers")
print(f"  mean predicted PD {out.pd_4m.mean():.4%}  (TEST observed {yte.mean():.4%})")
if out.pd_4m.mean() > 1.5 * yte.mean():
    print("\n  NOTE the live set scores materially riskier than TEST. That is")
    print("  consistent with the measured drift - the production bar admits a")
    print("  broader population in the scoring window - and is a reason to")
    print("  trust the PD RANKING more than the PD LEVEL.")

print("\n  by grade")
gb = out.groupby("grade", observed=True).agg(n=("pd_4m", "size"),
                                             mean_pd=("pd_4m", "mean"))
gb["share"] = gb.n / gb.n.sum()
print(gb.to_string(float_format=lambda v: f"{v:12.6f}"))

In [ ]:
o = out.sort_values("pd_4m").reset_index(drop=True)
o["cum_pd"] = o.pd_4m.expanding().mean()

print("  SELECTING THE SAFEST N - this is the table the limit engine uses")
print(f"  {'take':>12} {'share':>7} {'book mean PD':>13} "
      f"{'exposure':>14} {'expected loss':>14}")
marks  = [int(len(o) * f) for f in
          (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)]
marks += [n for n in (3_000_000,) if n <= len(o)]   # the stated business minimum
for n in sorted(set(m for m in marks if m > 0)):
    pdn, expo = o.cum_pd.iloc[n - 1], n * TICKET_TOMAN
    print(f"  {n:>12,} {n/len(o):>6.1%} {pdn:>12.4%} "
          f"{expo/1e12:>11,.1f} tn {expo*pdn/1e9:>11,.1f} bn")

In [ ]:
print("  LARGEST BOOK UNDER A PD CEILING")
print(f"  {'ceiling':>9} {'take':>12} {'share':>7} {'exposure':>14} "
      f"{'expected loss':>14}")
for ceil in (0.0025, 0.005, 0.0075, 0.010, 0.015, 0.020):
    # cum_pd rises monotonically, so the last index under the ceiling is the
    # largest book that satisfies it.
    under = np.flatnonzero(o.cum_pd.to_numpy() <= ceil)
    if len(under) == 0:
        print(f"  {ceil:>8.2%} {'-':>12} {'-':>7}   no subscriber qualifies")
        continue
    n = int(under[-1]) + 1
    expo = n * TICKET_TOMAN
    print(f"  {ceil:>8.2%} {n:>12,} {n/len(o):>6.1%} "
          f"{expo/1e12:>11,.1f} tn {expo*o.cum_pd.iloc[n-1]/1e9:>11,.1f} bn")

print(f"\n  Exposure assumes every approved subscriber draws the full "
      f"{TICKET_TOMAN:,} Toman line, and expected loss assumes a two-way bar")
print("  loses the WHOLE balance (LGD 100 pct). Both are deliberately")
print("  pessimistic. Treat them as a ceiling on the loss, not a forecast.")

---
## 12. The handover files

| file | what it is |
|---|---|
| `outputs/handover_scores.csv` | one row per live subscriber: `sbrp_id`, `pd_4m`, `grade`, plus `pay_cover` and `tenure_m` as **tie-break keys** — isotonic emits a step function, so many subscribers share one `pd_4m` and the cut needs a defensible order among them. This is what implementation consumes. |
| `outputs/metrics.csv` | the model comparison — in-sample, VALID and TEST — for the model-risk record. |

In [ ]:
# float_format matters more than it looks. By default pandas writes small
# floats in scientific notation - 1.573e-07 - and if that CSV is loaded
# into a VARCHAR column, "ORDER BY pd_4m" becomes a TEXT sort in which
# '1e-07' sorts after '0.0252', putting the SAFEST subscribers last. On a
# simulation of the real value range that turned a 0.082 pct book into a
# 1.348 pct one while still returning the right row COUNT. A fixed
# 12-decimal format cannot produce scientific notation, so the file is
# unambiguous however it is typed on the way in.
path = os.path.join(OUTDIR, "handover_scores.csv")
out.to_csv(path, index=False, float_format="%.12f")
print(f"  written to {path}  ({len(out):,} rows)")

metrics = pd.DataFrame(
    RES + [dict(r, model=r["model"] + " [VALID]") for r in RV] + RT)
mpath = os.path.join(OUTDIR, "metrics.csv")
metrics.to_csv(mpath, index=False)
print(f"  metrics written to {mpath}\n")
print(metrics.to_string(index=False, float_format=lambda v: f"{v:10.6f}"))

---
## What is not reported, and why

**Accuracy.** At a rate near 0.55% a model predicting "nobody defaults" scores
99.45%. It cannot distinguish a working model from an empty one.

**An out-of-time estimate.** The split is random by subscriber, so TEST measures
generalisation to other subscribers in the **same period**. Section 5 is what
speaks to the later period, and the drift it measures is real: revenue per
subscriber-month up 1.64x at the p90 between the two windows. Treat TEST AUC as
a ceiling.

**In-sample metrics as evidence.** Printed in section 8 for the train-to-valid
gap, which is a useful overfitting signal. Selection is on VALID.

**A single headline number for the book.** Section 11 is deliberately a curve,
because the answer depends on an appetite the business sets, not on the model.

### Two figures to carry into that decision

At the production bar the scoring window holds **9,344,723** subscribers, not
the 6,879,803 that was on record — the fixed 170,000 Toman bar is nominal, so it
has loosened on its own as revenue inflated past it. At 500,000 Toman that is a
**4,672 bn Toman** book against a **15,000 bn** target: still **3.21x** short,
needing roughly **1,605,000 Toman** per subscriber. That is a line-size decision.

And the expected loss depends on **when** you lend, not only on whom: a 4-month
exposure over months 7-10 measured **~0.55%**, while the same screen exposed over
months 1-4 measured **0.95%**. Lending into Nowruz is the expensive
configuration.